# Task 2 -- Data Discovery, Profiling and Cleaning

Reads the raw dev.to JSON produced by `01_extract.ipynb` from `data/raw/` (listing fields
plus `body_markdown`/`body_html` per article), profiles it, flags data-quality issues,
flattens the nested structure, cleans it -- including stripping the HTML/Liquid markup out
of the article body -- and writes `data/interim/cleaned.csv`.

> Run `01_extract.ipynb` first if `data/raw/` is empty -- this notebook picks up its most
> recent `devto_*.json` output automatically.


In [ ]:
import glob
import json
import os

import pandas as pd

RAW_DIR = os.path.join("..", "data", "raw")
raw_files = sorted(glob.glob(os.path.join(RAW_DIR, "devto_*.json")))
if not raw_files:
    raise FileNotFoundError(
        "No data/raw/devto_*.json found -- run 01_extract.ipynb first to produce one."
    )

raw_path = raw_files[-1]  # most recent
with open(raw_path, encoding="utf-8") as f:
    raw_records = json.load(f)

print(f"Loaded {len(raw_records)} raw records from {raw_path}")


## Profiling (raw, pre-flatten)

Column names, row/column counts, data types, null counts, unique values, and sample rows.

In [ ]:
raw_df = pd.json_normalize(raw_records)

print("Shape (rows, columns):", raw_df.shape)
print("\nColumns:", list(raw_df.columns))
print("\nDtypes:")
print(raw_df.dtypes)
print("\nNull counts:")
print(raw_df.isna().sum())

raw_df.head(3)


In [ ]:
# Profiling summary table deliverable: one row per column.
# raw_df has list/dict-valued columns (e.g. "tag_list"), which pandas' own
# .nunique() can't hash directly -- TypeError: unhashable type: 'list'.
# Fixed by hashing list/dict values through a JSON dump before counting.

def _hashable(v):
    if isinstance(v, (list, dict)):
        return json.dumps(v, sort_keys=True, default=str)
    return v

profile_rows = []
for col in raw_df.columns:
    col_series = raw_df[col]
    non_null = col_series.dropna()
    unique_count = len({_hashable(v) for v in non_null})
    profile_rows.append({
        "column": col,
        "dtype": str(col_series.dtype),
        "null_count": int(col_series.isna().sum()),
        "percent_null": round(100 * col_series.isna().mean(), 1),
        "unique_count": unique_count,
        "sample_value": non_null.iloc[0] if len(non_null) else None,
    })

profile_summary = pd.DataFrame(profile_rows)
profile_summary


## Issues found

- **Nested fields** -- `user` (dict) and `tag_list` (list) come back nested from the API.
  Decision: flatten via `standardize_devto_article()` below -- `user` -> `author`
  (name, falling back to username), `tag_list` -> `tags` (comma-joined string) and
  `topic` (first tag).
- **Duplicate articles** -- the same article can appear once per tag it was fetched under
  (we query several tags). Decision: drop duplicates by `url` after flattening.
- **HTML / Liquid markup noise in `body_markdown`** -- dev.to article bodies contain Liquid
  embed tags (`{% embed ... %}`, `{% github ... %}`), fenced code blocks, images, links and
  leftover HTML. Decision: strip these with `clean_markdown_to_text()` into a new
  `content_clean` column meant for NLP / keyword extraction, while keeping the original
  `content_markdown` / `content_html` untouched for reference.
- **Missing `cover_image` / `organization`** -- legitimately absent for many articles (not
  every author sets a cover image or belongs to an org). Decision: left null, not dropped.
- **Mixed / inconsistent date fields** -- dev.to sometimes only fills `published_timestamp`
  when `published_at` is absent. Decision: coerce to a real timestamp with
  `pd.to_datetime(..., errors="coerce")`; unparseable/missing values become `NaT` rather
  than being dropped.
- **A few articles may be missing body content** -- the per-article content fetch in Task 1
  skips (and logs) any article whose detail call fails, so `body_markdown` can be null for a
  small number of rows. Decision: left null, `content_clean` is also null for those rows.
- **List/dict-valued columns break naive `nunique()`** -- `tag_list` (and similar) are Python lists, which pandas can't hash directly (`TypeError: unhashable type: 'list'`). Decision: the profiling table above hashes list/dict values via `json.dumps()` before counting uniques, instead of dropping or stringifying the column outright.\n

## Flatten: map each raw article onto the pipeline's unified schema

In [ ]:
from typing import Any


def standardize_devto_article(raw: dict[str, Any]) -> dict[str, Any]:
    """Flattens one raw dev.to article dict onto the pipeline's unified content schema."""
    user = raw.get("user") or {}
    tag_list = raw.get("tag_list") or []
    if isinstance(tag_list, str):
        tag_list = [t.strip() for t in tag_list.split(",") if t.strip()]

    return {
        "external_id": f"devto:{raw.get('id')}",
        "title": (raw.get("title") or "").strip(),
        "source": "dev.to",
        "author": user.get("name") or user.get("username") or "",
        "published_date": raw.get("published_at") or raw.get("published_timestamp"),
        "url": raw.get("url"),
        "topic": tag_list[0] if tag_list else "",
        "tags": ", ".join(tag_list),
        "description": (raw.get("description") or "").strip(),
        "content_type": "article",
        "reading_time_minutes": raw.get("reading_time_minutes"),
        "reactions_count": raw.get("public_reactions_count"),
        "comments_count": raw.get("comments_count"),
        "cover_image": raw.get("cover_image"),
        # Carried straight over from Task 1's raw record -- cleaned up below.
        "content_markdown": raw.get("body_markdown"),
        "content_html": raw.get("body_html"),
        "content_clean": None,  # filled in the cleaning step
    }


standardized = [standardize_devto_article(r) for r in raw_records]
flat_df = pd.DataFrame(standardized)
print("Flattened shape:", flat_df.shape)
print("Rows with body content:", flat_df["content_markdown"].notna().sum())
flat_df.head(3)


## Clean: fix dtypes, handle nulls, drop duplicates, standardise text/dates, snake_case columns

Includes stripping the dev.to-flavoured Liquid/embed tags, fenced code blocks, images,
links and leftover HTML out of `content_markdown` so `content_clean` is safe, readable
plain text (for NLP / keyword extraction downstream).

In [ ]:
import re

# --- dev.to-flavoured Liquid/embed tags that show up in body_markdown ---
_LIQUID_TAG_RE = re.compile(r"\{%.*?%\}", flags=re.DOTALL)
_CODE_FENCE_RE = re.compile(r"```.*?```", flags=re.DOTALL)
_IMAGE_RE = re.compile(r"!\[[^\]]*\]\([^)]*\)")
_LINK_RE = re.compile(r"\[([^\]]+)\]\([^)]*\)")
_INLINE_CODE_RE = re.compile(r"`([^`]*)`")
_HEADING_RE = re.compile(r"^#{1,6}\s*", flags=re.MULTILINE)
_HR_RE = re.compile(r"^\s*-{3,}\s*$", flags=re.MULTILINE)
_BOLD_ITALIC_RE = re.compile(r"(\*\*|__)(.*?)\1|(\*|_)(.*?)\3", flags=re.DOTALL)
_HTML_TAG_RE = re.compile(r"<[^>]+>")
_BLANK_LINES_RE = re.compile(r"\n{3,}")


def clean_markdown_to_text(markdown):
    """Strips structural/markup noise out of a dev.to article's raw markdown."""
    if not markdown:
        return ""
    text = markdown
    text = _LIQUID_TAG_RE.sub("", text)
    text = _CODE_FENCE_RE.sub("", text)
    text = _IMAGE_RE.sub("", text)
    text = _LINK_RE.sub(r"\1", text)
    text = _INLINE_CODE_RE.sub(r"\1", text)
    text = _HEADING_RE.sub("", text)
    text = _HR_RE.sub("", text)
    text = _BOLD_ITALIC_RE.sub(lambda m: m.group(2) or m.group(4) or "", text)
    text = _HTML_TAG_RE.sub("", text)
    text = _BLANK_LINES_RE.sub("\n\n", text)
    return text.strip()


cleaned_df = flat_df.copy()

# Columns are already snake_case by construction; keep this explicit per the working rules.
cleaned_df.columns = [c.strip().lower().replace(" ", "_") for c in cleaned_df.columns]

# Drop duplicates: the same article can come back under more than one requested tag.
before = len(cleaned_df)
cleaned_df = cleaned_df.drop_duplicates(subset="url").reset_index(drop=True)
print(f"Dropped {before - len(cleaned_df)} duplicate rows (same url, multiple tags)")

# Quality filter: a record with no title or url is not usable downstream.
cleaned_df = cleaned_df[cleaned_df["title"].astype(bool) & cleaned_df["url"].astype(bool)]

# Dtypes / text / date standardisation.
cleaned_df["title"] = cleaned_df["title"].str.strip()
cleaned_df["description"] = cleaned_df["description"].fillna("").str.strip()
cleaned_df["tags"] = cleaned_df["tags"].fillna("")
cleaned_df["published_date"] = pd.to_datetime(cleaned_df["published_date"], errors="coerce", utc=True)
cleaned_df["reading_time_minutes"] = pd.to_numeric(cleaned_df["reading_time_minutes"], errors="coerce").fillna(0).astype(int)
cleaned_df["reactions_count"] = pd.to_numeric(cleaned_df["reactions_count"], errors="coerce").fillna(0).astype(int)
cleaned_df["comments_count"] = pd.to_numeric(cleaned_df["comments_count"], errors="coerce").fillna(0).astype(int)

# The whole point of this step: turn the raw markdown body into NLP-safe plain text.
cleaned_df["content_clean"] = cleaned_df["content_markdown"].apply(clean_markdown_to_text)

print("Cleaned shape:", cleaned_df.shape)
print("Rows with non-empty content_clean:", (cleaned_df["content_clean"].str.len() > 0).sum())
cleaned_df.head(3)


## Save the cleaned dataset

In [ ]:
INTERIM_DIR = os.path.join("..", "data", "interim")
os.makedirs(INTERIM_DIR, exist_ok=True)
cleaned_path = os.path.join(INTERIM_DIR, "cleaned.csv")

cleaned_df.to_csv(cleaned_path, index=False)
print(f"Saved -> {cleaned_path} ({len(cleaned_df)} rows)")
